# 12. Latents, I2AV references, decode и output contracts

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** проверить shapes/offsets, воспроизвести visual reference channels и разобрать encode/decode API

**Перед началом:** K6 03–04, 08 и 11. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/12_generation_utils.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §4.2.3, 5](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/algo/prepare_latents.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/prepare_latents.py)
- [`core/algo/visual_conditioning.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/visual_conditioning.py)
- [`core/algo/postprocess_video.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/postprocess_video.py)
- [`core/algo/postprocess_audio.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/postprocess_audio.py)
- [`core/algo/mux.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/mux.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','kandinsky-6-components'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '12_latents_conditioning_and_output'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Shape сам по себе не задаёт смысл

Video codec ожидает BCTHW, native DiT — BTHWC или packed THWC. Audio decoder ожидает BCA,
DiT — BAC или packed AC. `cu_seqlens` содержит границы последовательностей, а не всегда token counts:
у video это latent frames, которые после spatial patching разворачиваются в большее число tokens.
Native helpers создают reproducible CPU noise; межплатформенная RNG parity отдельно не предполагается.


In [ ]:
p=native('core.algo.prepare_latents');Bundle=native('core.types').LatentBundle
b=p.prepare_video_latents(2,3,4,4,4,42,'cpu',dtype=torch.float32)
b=p.prepare_audio_latents(b,7,4,42,'cpu',dtype=torch.float32)
b2=p.prepare_video_latents(2,3,4,4,4,42,'cpu',dtype=torch.float32)
assert torch.equal(b.video,b2.video)
positions=p.prepare_audio_rope_pos(b.audio_cu_seqlens)
assert positions.tolist()==list(range(7))*2
print('Packed shapes:',b.video.shape,b.audio.shape,'video offsets:',b.video_cu_seqlens)
metrics['packed_shapes']=[list(b.video.shape),list(b.audio.shape)]
metrics['audio_positions_reset_each_sequence']=True


## 2. Tail conditioning и token roles

I2AV reference может добавляться **во временной оси**, а не только в каналах.
K6 помечает generated/reference frame через channel mask и Token Role Embedding.
RoPE задаёт позиционный смысл reference; TRE — дополнительный обучаемый сигнал.
Проверим append и restore на настоящих tensor helpers. Сам restore не доказывает,
что reference уже повлиял на denoising других кадров: это задача learned attention.


In [ ]:
visual=native('core.algo.visual_conditioning')
video=torch.randn(1,3,4,4,4);refs=torch.randn(1,1,4,4,4)
appended,roles,generated=visual.append_visual_references(video,refs,1,3)
assert appended.shape==(1,4,4,4,4) and roles.reshape(1,-1).tolist()==[[0,0,0,1]]
cu=torch.tensor([0,4],dtype=torch.int32)
inp=visual.build_tail_condition(appended.clone(),refs,cu)
assert inp.shape[-1]==9 and inp[0,-1,...,-1].eq(1).all()
assert inp[0,:-1,...,-1].eq(0).all()
corrupted=appended+10;visual.restore_tail_references(corrupted,refs,cu)
assert torch.equal(corrupted[:,-1:],refs)
metrics['tail_condition_shape']=list(inp.shape);metrics['reference_restore_error']=0.
print('Tail condition shape:',inp.shape,'role ids:',roles)


## 3. Проверяем encode wrapper без выдуманного VAE API

Pinned custom Hunyuan encode возвращает tensor внутри tuple; Diffusers VAE может возвращать
posterior object. Нельзя заменять одно другим, не посмотрев return contract.
В учебном test double явно моделируем tuple[tensor]. Так проверяем layout/scaling wrapper,
а не качество настоящего Hunyuan reconstruction.


In [ ]:
from types import SimpleNamespace
class TupleEncoderDouble:
    config=SimpleNamespace(scaling_factor=.476986)
    def encode(self,frames):
        B,C,T,H,W=frames.shape
        return (torch.ones(B,16,(T-1)//4+1,H//8,W//8),)
encoded=p.encode_video(torch.zeros(2,3,9,32,32),TupleEncoderDouble())
assert encoded.shape==(6,4,4,16)
assert torch.allclose(encoded,torch.full_like(encoded,.476986))
metrics['encode_wrapper_output_shape']=list(encoded.shape)
print('Packed wrapper output:',encoded.shape)
show_source('core/algo/postprocess_video.py','postprocess_video')


## 4. Последний этап — mux

Decode возвращает video uint8 и audio PCM; mux принимает уже сгенерированные streams.
Container fps, sample rate, число frames/samples и trimming определяют длительность файла.
Рассинхрон может появиться на этом этапе даже при корректных neural outputs.
Поэтому сохраняйте raw durations до mux и после чтения MP4; существующий ручной протокол содержит команды.


## Самостоятельное исследование

1. Нарисуйте BCTHW→BTHWC→packed THWC и обратный путь для batch=2; подпишите offsets.
2. Добавьте два references first/last и проверьте role mask, restore и positional correspondence.
3. Подставьте posterior object вместо tuple[tensor] и сформулируйте нужный adapter, не меняя source snapshot.
4. Перед полноценным MP4 запишите video/audio durations и допуск меньше одного audio latent step.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете обнаружить shape/API mismatch до загрузки больших весов и объяснить,
какой helper меняет layout, какой нормализует значения, а какой только объединяет streams.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 11. Sampling: Euler, CFG, schedules и NFE](11_sampling_cfg_and_nfe.ipynb) · [13. Runtime: память, offload, cache и путь к Apple MLX →](13_runtime_memory_and_mlx.ipynb)
